<a href="https://colab.research.google.com/github/Zhalil24/BreastMRI-CNN-Classification/blob/main/EfficientNet_B0NewResult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import os
import copy
import shutil

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display


# ============================================================
# 2. CİHAZ AYARI VE DRIVE BAĞLAMA
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

drive.mount('/content/drive')


# ============================================================
# 3. DATASET RAR DOSYASINI KOPYALAMA VE AÇMA
# ============================================================

rar_source_path = "/content/drive/MyDrive/data_set_zip/breast_mri_dataset.rar"
rar_target_path = "/content/breast_mri_dataset.rar"

shutil.copy(rar_source_path, rar_target_path)

# -o+ aynı dosyalar varsa üstüne yazması için
os.system(f'unrar x -o+ "{rar_target_path}" /content/')


# ============================================================
# 4. ÇIKTI KLASÖRLERİ
# ============================================================

output_root = "/content/drive/MyDrive/breast_mri_efficientnet_outputs"
charts_dir = os.path.join(output_root, "charts")
tables_dir = os.path.join(output_root, "tables")

os.makedirs(output_root, exist_ok=True)
os.makedirs(charts_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

print(f"Çıktı klasörü: {output_root}")


# ============================================================
# 5. VERİ HAZIRLIĞI VE AUGMENTATION
# ============================================================

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])


# ============================================================
# 6. VERİ SETLERİNİ YÜKLEME
# ============================================================

data_dir = "/content/breast_mri_dataset"

# Eğitim için augmentation'lı dataset
train_dataset_full = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=train_transform
)

# Fold validation için augmentation'sız train dataset
# Aynı train klasörü kullanılıyor ama validation tarafında rastgele dönüşüm yapılmıyor.
train_dataset_eval = datasets.ImageFolder(
    os.path.join(data_dir, "train"),
    transform=val_test_transform
)

# Mevcut val dataset tanımı korunuyor.
val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "val"),
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(data_dir, "test"),
    transform=val_test_transform
)

class_names = train_dataset_full.classes
print("Sınıflar:", train_dataset_full.class_to_idx)

if len(class_names) == 2:
    display_class_names = ["Benign", "Malignant"]
else:
    display_class_names = class_names


# ============================================================
# 7. MODEL OLUŞTURMA
# ============================================================

def create_model():
    model = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
    )

    for param in model.parameters():
        param.requires_grad = False

    # Son blokları fine-tune ediyoruz.
    for i in range(3, len(model.features)):
        for param in model.features[i].parameters():
            param.requires_grad = True

    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 2)

    return model.to(device)


# ============================================================
# 8. OPTIMIZER
# ============================================================

def get_optimizer(model):
    params_groups = [
        {
            "params": model.features[3:4].parameters(),
            "lr": 1e-5
        },
        {
            "params": model.features[4:6].parameters(),
            "lr": 5e-5
        },
        {
            "params": model.features[6:].parameters(),
            "lr": 1e-4
        },
        {
            "params": model.classifier.parameters(),
            "lr": 1e-3
        }
    ]

    optimizer = optim.Adam(params_groups)
    return optimizer


criterion = nn.CrossEntropyLoss()


# ============================================================
# 9. EĞİTİM VE VALIDATION FONKSİYONLARI
# ============================================================

def train_one_epoch(model, loader, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0

    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


def validate_one_epoch(model, loader):
    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels.data).item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


# ============================================================
# 10. DETAYLI DEĞERLENDİRME FONKSİYONU
# ============================================================

def evaluate_model(model, loader):
    model.eval()

    y_true = []
    y_pred = []
    y_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)

            _, preds = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0)
    }

    try:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probs)
    except ValueError:
        metrics["roc_auc"] = np.nan

    return metrics, y_true, y_pred, y_probs


# ============================================================
# 11. GRAFİK FONKSİYONLARI
# ============================================================

def plot_accuracy(history, fold_no, save_path):
    epochs_range = range(1, len(history["train_acc"]) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history["train_acc"],
        marker="o",
        label="Train Accuracy"
    )
    plt.plot(
        epochs_range,
        history["val_acc"],
        marker="o",
        label="Validation Accuracy"
    )

    plt.title(f"Fold {fold_no}/5 Accuracy Grafiği")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_loss(history, fold_no, save_path):
    epochs_range = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history["train_loss"],
        marker="o",
        label="Train Loss"
    )
    plt.plot(
        epochs_range,
        history["val_loss"],
        marker="o",
        label="Validation Loss"
    )

    plt.title(f"Fold {fold_no}/5 Loss Grafiği")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_confusion_matrix_graph(labels, preds, title, save_path):
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=display_class_names,
        yticklabels=display_class_names
    )

    plt.title(title)
    plt.ylabel("Gerçek")
    plt.xlabel("Tahmin")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_roc_curve_graph(labels, probs, title, save_path):
    plt.figure(figsize=(8, 6))

    try:
        fpr, tpr, _ = roc_curve(labels, probs)
        auc_value = roc_auc_score(labels, probs)

        plt.plot(
            fpr,
            tpr,
            lw=2,
            label=f"ROC Curve - AUC = {auc_value:.4f}"
        )

        plt.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            lw=2,
            label="Random Classifier"
        )

    except ValueError:
        plt.text(
            0.5,
            0.5,
            "ROC hesaplanamadı.\nValidation setinde tek sınıf olabilir.",
            ha="center",
            va="center"
        )

    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


# ============================================================
# 12. 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

targets = np.array(train_dataset_full.targets)

fold_results = []
fold_summary_rows = []
epoch_history_rows = []
output_file_rows = []

best_overall_val_acc = -1
best_model = None
last_fold_model = None

epochs = 10
batch_size = 32

# Eski kodun davranışına dönmek istersen bunu "last_fold" yapabilirsin.
# Tavsiye edilen: "best_fold"
FINAL_MODEL_POLICY = "best_fold"

print("--- 5-Fold Eğitim Başlıyor ---")

for fold_no, (train_idx, val_idx) in enumerate(
    skf.split(np.zeros(len(targets)), targets),
    start=1
):
    print("\n" + "=" * 20)
    print(f"Fold {fold_no}/5")
    print("=" * 20)

    train_sub = Subset(train_dataset_full, train_idx)

    # Önemli:
    # Validation için augmentation'sız dataset kullanıyoruz.
    val_sub = Subset(train_dataset_eval, val_idx)

    train_loader = DataLoader(
        train_sub,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_sub,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = create_model()
    optimizer = get_optimizer(model)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_epoch_val_acc = -1
    best_epoch_no = 1
    best_epoch_state = None

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer
        )

        val_loss, val_acc = validate_one_epoch(
            model,
            val_loader
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        epoch_history_rows.append({
            "Fold": fold_no,
            "Epoch": epoch + 1,
            "Train_Loss": train_loss,
            "Train_Accuracy": train_acc,
            "Validation_Loss": val_loss,
            "Validation_Accuracy": val_acc
        })

        print(
            f"Epoch {epoch+1}/{epochs}: "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_epoch_val_acc:
            best_epoch_val_acc = val_acc
            best_epoch_no = epoch + 1
            best_epoch_state = copy.deepcopy(model.state_dict())

    # Fold içindeki en iyi epoch ağırlıkları yükleniyor.
    if best_epoch_state is not None:
        model.load_state_dict(best_epoch_state)

    fold_results.append(history)

    # Fold validation detaylı metrikleri
    val_metrics, val_true, val_pred, val_probs = evaluate_model(
        model,
        val_loader
    )

    print(f"\nFold {fold_no} Validation Performansı")
    print(f"Best Epoch: {best_epoch_no}")
    print(f"Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"Precision: {val_metrics['precision']:.4f}")
    print(f"Recall:    {val_metrics['recall']:.4f}")
    print(f"F1-Score:  {val_metrics['f1_score']:.4f}")
    print(f"ROC-AUC:   {val_metrics['roc_auc']:.4f}")

    # Her fold için grafik dosya yolları
    accuracy_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_accuracy.png"
    )

    loss_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_loss.png"
    )

    cm_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_confusion_matrix.png"
    )

    roc_path = os.path.join(
        charts_dir,
        f"fold_{fold_no}_roc_curve.png"
    )

    # Her fold için 4 grafik
    plot_accuracy(
        history,
        fold_no,
        accuracy_path
    )

    plot_loss(
        history,
        fold_no,
        loss_path
    )

    plot_confusion_matrix_graph(
        val_true,
        val_pred,
        title=f"Fold {fold_no}/5 Validation Confusion Matrix",
        save_path=cm_path
    )

    plot_roc_curve_graph(
        val_true,
        val_probs,
        title=f"Fold {fold_no}/5 Validation ROC-AUC Grafiği",
        save_path=roc_path
    )

    fold_summary_rows.append({
        "Fold": fold_no,
        "Best_Epoch": best_epoch_no,
        "Validation_Accuracy": val_metrics["accuracy"],
        "Validation_Precision": val_metrics["precision"],
        "Validation_Recall": val_metrics["recall"],
        "Validation_F1_Score": val_metrics["f1_score"],
        "Validation_ROC_AUC": val_metrics["roc_auc"],
        "Accuracy_Graph": accuracy_path,
        "Loss_Graph": loss_path,
        "Confusion_Matrix_Graph": cm_path,
        "ROC_Curve_Graph": roc_path
    })

    output_file_rows.extend([
        {
            "Output_Type": "Fold Accuracy Graph",
            "Fold": fold_no,
            "File_Path": accuracy_path
        },
        {
            "Output_Type": "Fold Loss Graph",
            "Fold": fold_no,
            "File_Path": loss_path
        },
        {
            "Output_Type": "Fold Confusion Matrix Graph",
            "Fold": fold_no,
            "File_Path": cm_path
        },
        {
            "Output_Type": "Fold ROC Curve Graph",
            "Fold": fold_no,
            "File_Path": roc_path
        }
    ])

    last_fold_model = copy.deepcopy(model)

    if val_metrics["accuracy"] > best_overall_val_acc:
        best_overall_val_acc = val_metrics["accuracy"]
        best_model = copy.deepcopy(model)

print("\n--- Tüm Foldlar Başarıyla Tamamlandı ---")


# ============================================================
# 13. FOLD TABLOLARI
# ============================================================

fold_summary_df = pd.DataFrame(fold_summary_rows)
epoch_history_df = pd.DataFrame(epoch_history_rows)

print("\n--- FOLD ÖZET TABLOSU ---")
display(fold_summary_df)

print("\n--- EPOCH GEÇMİŞ TABLOSU ---")
display(epoch_history_df)


# ============================================================
# 14. TEST SETİ ÜZERİNDE FINAL DEĞERLENDİRME
# ============================================================

print("\n--- Test Seti Üzerinde Final Değerlendirme ---")

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

if FINAL_MODEL_POLICY == "last_fold":
    final_model = last_fold_model
    final_model_name = "Last Fold Model"
else:
    final_model = best_model
    final_model_name = "Best Fold Model"

test_metrics, y_true, y_pred, y_probs = evaluate_model(
    final_model,
    test_loader
)

print(f"Final Model: {final_model_name}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")

final_test_df = pd.DataFrame([{
    "Final_Model": final_model_name,
    "Test_Accuracy": test_metrics["accuracy"],
    "Test_Precision": test_metrics["precision"],
    "Test_Recall": test_metrics["recall"],
    "Test_F1_Score": test_metrics["f1_score"],
    "Test_ROC_AUC": test_metrics["roc_auc"]
}])

print("\n--- FINAL TEST TABLOSU ---")
display(final_test_df)


# ============================================================
# 15. MEVCUT ORTALAMA ACCURACY GRAFİĞİ
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

epochs_count = len(fold_results[0]["train_acc"])

sum_train_acc = np.zeros(epochs_count)
sum_val_acc = np.zeros(epochs_count)

for history_fold in fold_results:
    sum_train_acc += np.array(history_fold["train_acc"])
    sum_val_acc += np.array(history_fold["val_acc"])

avg_train_acc = sum_train_acc / len(fold_results)
avg_val_acc = sum_val_acc / len(fold_results)

average_accuracy_path = os.path.join(
    charts_dir,
    "average_training_vs_validation_accuracy.png"
)

plt.figure(figsize=(9, 6))
plt.plot(
    range(1, epochs_count + 1),
    avg_train_acc,
    marker="o",
    label="Average Train Acc"
)
plt.plot(
    range(1, epochs_count + 1),
    avg_val_acc,
    marker="o",
    label="Average Val Acc"
)

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(average_accuracy_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Average Accuracy Graph",
    "Fold": "All",
    "File_Path": average_accuracy_path
})


# ============================================================
# 16. MEVCUT FINAL TEST CONFUSION MATRIX
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

final_test_cm_path = os.path.join(
    charts_dir,
    "final_test_confusion_matrix.png"
)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=display_class_names,
    yticklabels=display_class_names
)
plt.title("Confusion Matrix")
plt.ylabel("Gerçek")
plt.xlabel("Tahmin")
plt.tight_layout()
plt.savefig(final_test_cm_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Final Test Confusion Matrix Graph",
    "Fold": "Final Test",
    "File_Path": final_test_cm_path
})


# ============================================================
# 17. MEVCUT FINAL TEST ROC CURVE
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

final_test_roc_path = os.path.join(
    charts_dir,
    "final_test_roc_curve.png"
)

fpr, tpr, _ = roc_curve(y_true, y_probs)
test_auc = roc_auc_score(y_true, y_probs)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr,
    tpr,
    lw=2,
    label=f"ROC curve area = {test_auc:.2f}"
)
plt.plot(
    [0, 1],
    [0, 1],
    lw=2,
    linestyle="--"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(final_test_roc_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

output_file_rows.append({
    "Output_Type": "Final Test ROC Curve Graph",
    "Fold": "Final Test",
    "File_Path": final_test_roc_path
})


# ============================================================
# 18. TÜM SONUÇLARI EXCEL DOSYASINA KAYDETME
# ============================================================

output_files_df = pd.DataFrame(output_file_rows)

excel_path = os.path.join(
    tables_dir,
    "efficientnet_breast_mri_model_results.xlsx"
)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    fold_summary_df.to_excel(
        writer,
        sheet_name="Fold_Summary",
        index=False
    )

    epoch_history_df.to_excel(
        writer,
        sheet_name="Epoch_History",
        index=False
    )

    final_test_df.to_excel(
        writer,
        sheet_name="Final_Test",
        index=False
    )

    output_files_df.to_excel(
        writer,
        sheet_name="Output_Files",
        index=False
    )

print("\n--- KAYIT İŞLEMİ TAMAMLANDI ---")
print(f"Grafikler klasörü: {charts_dir}")
print(f"Excel dosyası: {excel_path}")

print("\nÜretilen çıktı dosyaları:")
display(output_files_df)